# 1D-CNN Classifier. Loading and evaluating a model

In [1]:
import sys
print(sys.executable)

C:\Users\nxf93627\Projects\PQD-Classifier\.venv\Scripts\python.exe


In [2]:
import config
import dataset
import models
import training
import utils

In [3]:
settings = config.load_config('base_config.yaml')
settings

Reading config yaml


{'model': {'name': 'LiteratureCNN',
  'kernel_size': 3,
  'stride': 1,
  'schedule': 'scheduler',
  'optimizer': 'nadam',
  'loss_function': 'cross-entropy'},
 'training': {'batch_size': 64,
  'epochs': 43,
  'learning_rate': 0.001,
  'seed': 42},
 'dataset': {'filename': 'C:\\Users\\nxf93627\\PQD Classifier\\data\\16pqd_480pattern_50hz_10cycle_noNoise.mat',
  'seed_random': 42,
  'test_percentage': 0.1}}

In [4]:
X_train, Y_train, X_test, Y_test, encoder, metadata = dataset.loadMatlabDataset(settings['dataset'])

model_cfg = settings['model']
model_cfg.update({'input_shape': (metadata['observations'], 1), 'outputs': metadata['classes']})
model_cfg

Loading dataset from C:\Users\nxf93627\PQD Classifier\data\16pqd_480pattern_50hz_10cycle_noNoise.mat
Successfully read 7680 samples
Encoding labels with OneHot
Successfully encoded 16 categories
Splitted training (0.9) and testing sets (0.1)


{'name': 'LiteratureCNN',
 'kernel_size': 3,
 'stride': 1,
 'schedule': 'scheduler',
 'optimizer': 'nadam',
 'loss_function': 'cross-entropy',
 'input_shape': (640, 1),
 'outputs': 16}

In [5]:
model = models.build_model(settings['model'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                      │ (None, 638, 32)             │             128 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_1 (Conv1D)                    │ (None, 636, 32)             │           3,104 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d (MaxPooling1D)         │ (None, 634, 32)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_2 (Conv1D)                    │ (None, 632, 64)             │           6,208 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_3 (Conv1D)                    │ (None, 630, 64)             │          12,352 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_1 (MaxPooling1D)       │ (None, 628, 64)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_4 (Conv1D)                    │ (None, 626, 128)            │          24,704 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_5 (Conv1D)                    │ (None, 624, 128)            │          49,280 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_max_pooling1d                 │ (None, 128)                 │               0 │
│ (GlobalMaxPooling1D)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 128)                 │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │          33,024 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 128)                 │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 16)                  │           2,064 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 164,784 (643.69 KB)

 Trainable params: 164,272 (641.69 KB)

 Non-trainable params: 512 (2.00 KB)

In [9]:
model.load_weights(utils.MODELS_DIR / 'run_08_15-13_56' / 'model.weights.h5')
model = training.evaluate(model, X_test, Y_test, metadata['categories'])

24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step
compile_metrics: 80.46875
accuracy: 0.7955729166666666
macro_f1: 0.7979322217493006
weighted_f1: 0.7979322217493007
classification_report:                         precision    recall  f1-score   support

               Flicker       0.94      0.65      0.77        48
     Flicker+Harmonics       1.00      1.00      1.00        48
           Flicker+Sag       0.96      0.94      0.95        48
         Flicker+Swell       0.73      0.98      0.84        48
             Harmonics       0.92      1.00      0.96        48
   Impulsive Transient       1.00      0.69      0.81        48
          Interruption       0.52      0.50      0.51        48
Interruption+Harmonics       0.44      0.48      0.46        48
                Normal       0.83      1.00      0.91        48
                 Notch       0.98      0.98      0.98        48
 Oscillatory transient       1.00      0.92      0.96        48
                   Sag       0.49      0.35      0.4

C:\Users\nxf93627\Projects\PQD-Classifier\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
